In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
// var db = OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble2D");
//var db = OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble3D");
var db = OpenOrCreateDatabase(@"P:\hpccluster\DropletRebound_Gauthier");

In [ ]:
var sessions = db.Sessions;
sessions

#0: DropletRebound_Gauthier	DropletReboundGauthier_8x8x8AMR0_k3_ReI4_2cores_LineSearch_restart2*	07/17/2024 11:34:50	ed718c6f...
#1: DropletRebound_Gauthier	DropletReboundGauthier_8x8x8AMR0_k3_ReI4_2cores2threads_restart1*	07/03/2024 09:42:01	b3d24b15...
#2: DropletRebound_Gauthier	DropletReboundGauthier_8x8x8AMR0_k3_ReI4_2threads_restart1*	07/01/2024 09:47:24	6e34a603...
#3: DropletRebound_Gauthier	DropletReboundGauthier_8x8x8AMR0_k3_ReI4_2cores_restart1*	07/01/2024 09:45:33	1a8cfde2...
#4: DropletRebound_Gauthier	DropletReboundGauthier_8x8x8AMR0_k3_ReI4_restart1*	07/01/2024 09:08:59	57e8f3ff...
#5: DropletRebound_Gauthier	DropletReboundGauthier_8x12x8AMR1_k2_ReI4_singleCore_restart3*	06/21/2024 15:25:14	80683059...
#6: DropletRebound_Gauthier	DropletReboundGauthier_8x8x8AMR0_k3_ReI4*	06/20/2024 14:39:47	89340185...
#7: DropletRebound_Gauthier	DropletReboundGauthier_8x8x8AMR1_k3_ReI4_restart6_DebugRun*	06/13/2024 13:40:27	b02a8317...
#8: DropletRebound_Gauthier	DropletReboundGauthier_

In [ ]:
var sess = sessions.Pick(3);
sess

DropletRebound_Gauthier	DropletReboundGauthier_8x8x8AMR0_k3_ReI4_2cores_restart1*	07/01/2024 09:45:33	1a8cfde2...

In [ ]:
sess.Timesteps.Skip(30).Take(7) //.Export().WithSupersampling(2).Do()

C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletRebound_Gauthier__DropletReboundGauthier_8x8x8AMR0_k3_ReI4_2cores_restart1__1a8cfde2-27a6-4c58-b4ed-0607a243731a

## Setting up for some checks

In [ ]:
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.LevelSetTools;

In [ ]:
var timesteps = sess.Timesteps.Skip(30).Take(7);
timesteps

#0:  { Time-step: 48; Physical time: 0.0012s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }
#1:  { Time-step: 49; Physical time: 0.001225s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }
#2:  { Time-step: 50; Physical time: 0.00125s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }
#3:  { Time-step: 51; Physical time: 0.001275s; Fields: Phi, Phi

In [ ]:
var tStep = sess.Timesteps.First(); 
tStep

 { Time-step: 18; Physical time: 0.00045000000000000015s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }

In [ ]:
// SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
// GridData grdData = (GridData)phi.GridDat; 
// LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
// levelSet.Acc(1.0, phi); 
// LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
// LsTrk.UpdateTracker(tStep.PhysicalTime);

In [ ]:
// SinglePhaseField phiDG = (SinglePhaseField)tStep.GetField("PhiDG");
// SinglePhaseField phiCP = new SinglePhaseField(new Basis(grdData, phiDG.Basis.Degree + 1), "PhiCP");

// ContinuityProjection contiProj = new ContinuityProjection(
//     phiCP.Basis,
//     phiDG.Basis,
//     grdData,
//     ContinuityProjectionOption.ConstrainedDG);

In [ ]:
// contiProj.MakeContinuous(phiDG, phiCP, LsTrk.Regions.GetCutCellMask(), LsTrk.Regions.GetLevelSetWing(0, +1).VolumeMask);
// levelSet = new LevelSet(phiCP.Basis, "levelSet");
// levelSet.Acc(1.0, phiCP); 
// LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
// LsTrk.UpdateTracker(tStep.PhysicalTime);

In [ ]:
// int order = 2 * phi.Basis.Degree + 1;
// XQuadSchemeHelper SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;

In [ ]:
// SpeciesId spcId_A = LsTrk.GetSpeciesId("A");
// SpeciesId spcId_B = LsTrk.GetSpeciesId("B");

In [ ]:
public static LevelSetTracker GetLsTrkForTimestep(ITimestepInfo _tStep, XQuadFactoryHelper.MomentFittingVariants _mfv) {

    SinglePhaseField ts_phi = (SinglePhaseField)_tStep.GetField("Phi");
    GridData ts_grdData = (GridData)ts_phi.GridDat; 
    LevelSet ts_levelSet = new LevelSet(ts_phi.Basis, "levelSet");
    ts_levelSet.Acc(1.0, ts_phi); 
    LevelSetTracker ts_LsTrk = new LevelSetTracker(ts_grdData, _mfv,  1, new string[] {"A", "B"}, ts_levelSet);
    ts_LsTrk.UpdateTracker(_tStep.PhysicalTime);

    return ts_LsTrk;
}

In [ ]:
var mfv = XQuadFactoryHelper.MomentFittingVariants.Saye; //sess.GetControl().CutCellQuadratureType;
mfv

Saye

In [ ]:
public static int GetXNSEquadOrder(int _degU, XQuadFactoryHelper.MomentFittingVariants _mfv, bool _conv = true) {

    //QuadOrder
     int quadOrder = _degU * (_conv ? 3 : 2);
     if(_mfv == XQuadFactoryHelper.MomentFittingVariants.Saye) {
         //See remarks
         quadOrder *= 2;
         quadOrder += 1;
     } 

     return quadOrder;
}

## Checking Integral properties

In [ ]:
using BoSSS.Foundation.XDG.Quadrature.HMF;
using BoSSS.Solution.Tecplot;

In [ ]:
public static VectorField<SinglePhaseField> GetTestField(GridData grdDat) {

    Basis ScalarBasis = new Basis(grdDat, 0); 

    SinglePhaseField testFieldX = new SinglePhaseField(ScalarBasis);
    Func<double[], double> fx = (X => 1.0);
    //Func<double[], double> fx = (X => X[0] + X[1] + X[2]);
    testFieldX.ProjectField(fx);
    SinglePhaseField testFieldY = new SinglePhaseField(ScalarBasis);
    Func<double[], double> fy = (X => 1.0);
    //Func<double[], double> fy = (X => X[0] + X[1] + X[2]);
    testFieldY.ProjectField(fy);
    SinglePhaseField testFieldZ = new SinglePhaseField(ScalarBasis);
    Func<double[], double> fz = (X => 1.0);
    //Func<double[], double> fz = (X => X[0] + X[1] - 2.0*X[2]);
    testFieldZ.ProjectField(fz);

    if (grdDat.SpatialDimension == 2) {
        return new VectorField<SinglePhaseField>(new SinglePhaseField[] {testFieldX, testFieldY});
    } else {
        return new VectorField<SinglePhaseField>(new SinglePhaseField[] {testFieldX, testFieldY, testFieldZ});
    }

}

check Gauss on the whole cutCell domain at one specific timestep

In [ ]:
public static double GetGaussErrorOnInterface(ITimestepInfo _tStep, XQuadFactoryHelper.MomentFittingVariants _mfv) {

    var LsTrk = GetLsTrkForTimestep(_tStep, _mfv);
    CellMask CCmask = LsTrk.Regions.GetCutCellMask();
    int degU = _tStep.GetField("VelocityX").Basis.Degree;
    int order = GetXNSEquadOrder(degU, _mfv);
    XQuadSchemeHelper schemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;

    var testField = GetTestField(LsTrk.GridDat);

    double gaussErrOnDomain = 0.0;

    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetLevelSetquadScheme(0, CCmask).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            var LsNormals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, length);

            for (int d = 0; d < D; d++) {

                var fEval_d = MultidimensionalArray.Create(length, qN);
                testField[d].Evaluate(i0, length, QR.Nodes, fEval_d);

                for (int i = 0; i < length; i++) {
                    for (int qn = 0; qn < qN; qn++) {
                        EvalResult[i, qn, 0] += fEval_d[i, qn] * LsNormals[i, qn, d];
                    }
                }
            }

        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                gaussErrOnDomain += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();

    return gaussErrOnDomain;
}

check Gauss on the whole cutCell domain for a range of timesteps

In [ ]:
bool checkRange = true;

List<double> GaussOnDomainOverTime = new List<double>();
if (checkRange) {
    foreach(var ts in timesteps) {

        double gaussOnDomain = GetGaussErrorOnInterface(ts, mfv);

        GaussOnDomainOverTime.Add(gaussOnDomain);
    }
}

In [ ]:
GaussOnDomainOverTime

index,value
0,-3.890013483395944E-11
1,2.5184334345537684E-11
2,-6.518314865953817E-10
3,-2.487099911345356E-09
4,-1.0046317420143318E-08
5,-2.0344084342873088E-08
6,-1.875228747984389E-08


In [ ]:
public static (double error, double gaussInVolume, double gaussAtInterface, double gaussAtEdges) CheckGaussForCell(int jCell, LevelSetTracker LsTrk, SpeciesId spcId_A, VectorField<SinglePhaseField> testField, int order) {

    GridData grdDat = LsTrk.GridDat;
    BitArray cellIntDomBA = new BitArray(grdDat.Cells.NoOfLocalUpdatedCells);
    cellIntDomBA[jCell] = true;
    CellMask cellIntDom = new CellMask(grdDat, cellIntDomBA); //, MaskType.Geometrical);

    XQuadSchemeHelper schemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;

    // ========================================================
    double gaussInVolume = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetLevelSetquadScheme(0, cellIntDom, order).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            for (int d = 0; d < D; d++) {

                var fGrad_d = MultidimensionalArray.Create(length, qN, D);
                testField[d].EvaluateGradient(i0, length, QR.Nodes, fGrad_d, 0, 0.0);

                for (int i = 0; i < length; i++) {
                    for (int qn = 0; qn < qN; qn++) {
                        EvalResult[i, qn, 0] += fGrad_d[i, qn, d];
                    }
                }
            }
            //EvalResult.SetAll(1.0);
        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                gaussInVolume += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();
    if(gaussInVolume != 0.0)
        Console.WriteLine($"gauss in volume not zero! it is {gaussInVolume}");

    // =========================================================
    double gaussAtInterface = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetLevelSetquadScheme(0, cellIntDom).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            var LsNormals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, length);        // TODO change direction for spcId_B

            for (int d = 0; d < D; d++) {

                var fEval_d = MultidimensionalArray.Create(length, qN);
                testField[d].Evaluate(i0, length, QR.Nodes, fEval_d);

                for (int i = 0; i < length; i++) {
                    for (int qn = 0; qn < qN; qn++) {
                        EvalResult[i, qn, 0] += fEval_d[i, qn] * LsNormals[i, qn, d];
                    }
                }
            }
            // EvalResult.SetAll(1.0);
            
        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                gaussAtInterface += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();

    // =========================================================
    BitArray edgeIntDomBA = new BitArray(grdDat.iGeomEdges.Count);
    var jCell2Edges = grdDat.Cells.Cells2Edges[jCell];
    foreach (int iE in jCell2Edges) {
        if (iE < 0) 
            edgeIntDomBA[(-iE) - 1] = true;
        else 
            edgeIntDomBA[iE - 1] = true;
    }
    EdgeMask edgeIntDom = new EdgeMask(grdDat, edgeIntDomBA);

    double gaussAtCutEdges = 0.0;
    EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetEdgeQuadScheme(spcId_A, IntegrationDomain: edgeIntDom).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            for (int d = 0; d < D; d++) {

                var fEvalEdgeIN_d = MultidimensionalArray.Create(length, qN);
                var fEvalEdgeOUT_d = MultidimensionalArray.Create(length, qN);
                testField[d].EvaluateEdge(i0, length, QR.Nodes, fEvalEdgeIN_d, fEvalEdgeOUT_d);

                for (int i = 0; i < length; i++) {
                    double[] EdgeNormal = LsTrk.GridDat.Edges.NormalsForAffine.ExtractSubArrayShallow(i0 + i, -1).To1DArray();  
                    if (LsTrk.GridDat.Edges.CellIndices[i0 + i, 1] == jCell) {
                        // Console.WriteLine($"jCell {jCell} OUT on edge {i0 + i} - change direction");
                        EdgeNormal.ScaleV(-1.0);
                    }
                    // Console.WriteLine($"Edge normal: ({EdgeNormal[0]}, {EdgeNormal[1]})");

                    for (int qn = 0; qn < qN; qn++) {
                        EvalResult[i, qn, 0] += fEvalEdgeIN_d[i, qn] * EdgeNormal[d];
                    }
                }
            }
            // EvalResult.SetAll(1.0);

        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                // Console.WriteLine($"edge {i0 + i}");
                gaussAtCutEdges += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();

    // =========================================================
    double error = gaussInVolume - gaussAtInterface - gaussAtCutEdges;

    return (error, gaussInVolume, gaussAtInterface, gaussAtCutEdges);

}

In [ ]:
List<double> GaussInCutCellsOverTime = new List<double>();
if (checkRange) {
    foreach(var ts in timesteps) {

        var ts_LsTrk = GetLsTrkForTimestep(ts, mfv);

        int degU = ts.GetField("VelocityX").Basis.Degree;
        int ts_order = GetXNSEquadOrder(degU, mfv);
       
        var testField = GetTestField(ts_LsTrk.GridDat);

        double totalGaussErrorInCC = 0.0;
        SinglePhaseField gaussErrorField_ts = new SinglePhaseField(new Basis(ts_LsTrk.GridDat, 0), "gaussErrorAbs");
        foreach (int jC in ts_LsTrk.Regions.GetCutCellMask().ItemEnum) {
            var result = CheckGaussForCell(jC, ts_LsTrk, ts_LsTrk.GetSpeciesId("A"), testField, ts_order);
            gaussErrorField_ts.SetMeanValue(jC, result.error.Abs());
            totalGaussErrorInCC += result.error.Abs();
        }
        DGField[] plotFields = new DGField[] { (SinglePhaseField)ts.GetField("Phi"), gaussErrorField_ts };
        Tecplot("GaussErrors3D_" + ts.TimeStepNumber.MajorNumber, tStep.PhysicalTime, 3, plotFields);
        
        GaussInCutCellsOverTime.Add(totalGaussErrorInCC);
    }
}

In [ ]:
GaussInCutCellsOverTime

index,value
0,1.8338929212255095E-10
1,1.0242567269348881E-10
2,9.082554834799326E-10
3,2.6511216476958606E-09
4,1.0736159094816412E-08
5,2.036787290032826E-08
6,1.878674732624041E-08


check gauss on cut cells in timestep tStep

In [ ]:
var LsTrk = GetLsTrkForTimestep(tStep, mfv);

var testField = GetTestField(LsTrk.GridDat);

int degU = tStep.GetField("VelocityX").Basis.Degree;
int order = GetXNSEquadOrder(degU, mfv);
order

19

In [ ]:
double totalGaussError = 0.0;
SinglePhaseField gaussErrorField = new SinglePhaseField(new Basis(LsTrk.GridDat, 0), "gaussError");
foreach (int jC in LsTrk.Regions.GetCutCellMask().ItemEnum) {
    var result = CheckGaussForCell(jC, LsTrk, LsTrk.GetSpeciesId("A"), testField, order);
    double resErr = result.error;
    if (resErr.Abs() > 1.0e-10) {
        Console.WriteLine($"Gauss error in Cell {jC}: {resErr}");
    }
    gaussErrorField.SetMeanValue(jC, resErr);
    totalGaussError += resErr.Abs();
}
totalGaussError

Gauss error in Cell 288: 1.038127164226654E-08
Gauss error in Cell 344: 2.6216452134432943E-09
Gauss error in Cell 352: 2.381026395691626E-08


3.696924354173709E-08

In [ ]:
int jCell = 288;
CheckGaussForCell(jCell, LsTrk, LsTrk.GetSpeciesId("A"), testField, order)

Item1,Item2,Item3,Item4
1.038127164226654E-08,0,-1.851603844731282E-07,1.7477911283086165E-07


In [ ]:
public static (double error, double stokesAtInterface, double stokesAtEdges) CheckStokesForCell(int jCell, LevelSetTracker LsTrk, SpeciesId spcId_A, VectorField<SinglePhaseField> testField, int order) {

    GridData grdDat = LsTrk.GridDat;
    BitArray cellIntDomBA = new BitArray(grdDat.Cells.NoOfLocalUpdatedCells);
    cellIntDomBA[jCell] = true;
    CellMask cellIntDom = new CellMask(grdDat, cellIntDomBA); //, MaskType.Geometrical);

    XQuadSchemeHelper schemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;

    // ========================================================
    double stokesAtInterface = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetLevelSetquadScheme(0, cellIntDom, order).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            var LsNormals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, length);

            var curv = MultidimensionalArray.Create(length, qN);                              
            ((LevelSet)LsTrk.LevelSets[0]).EvaluateTotalCurvature(i0, length, QR.Nodes, curv);
            // curv.Scale(0.5);    // mean curvature

            for (int d = 0; d < D; d++) {

                var fEval_d = MultidimensionalArray.Create(length, qN);
                testField[d].Evaluate(i0, length, QR.Nodes, fEval_d);

                for (int i = 0; i < length; i++) {
                    for (int qn = 0; qn < qN; qn++) {  
                        EvalResult[i, qn, 0] += curv[i, qn] * LsNormals[i, qn, d] * fEval_d[i, qn];
                    }
                }
            }

        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                stokesAtInterface += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();
    
    // =========================================================
    BitArray edgeIntDomBA = new BitArray(grdDat.iGeomEdges.Count);
    var jCell2Edges = grdDat.Cells.Cells2Edges[jCell];
    foreach (int iE in jCell2Edges) {
        if (iE < 0) 
            edgeIntDomBA[(-iE) - 1] = true;
        else 
            edgeIntDomBA[iE - 1] = true;
    }
    EdgeMask edgeIntDom = new EdgeMask(grdDat, edgeIntDomBA);
    
    var testFactory = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadFactoryHelper.GetSurfaceElement_BoundaryRuleFactory(0, LsTrk.GridDat.Grid.RefElements[0]);
    EdgeQuadratureScheme SurfaceElement_BoundaryEdge = new EdgeQuadratureScheme(testFactory, edgeIntDom);
    
    double stokesAtEdges = 0.0;
    EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        SurfaceElement_BoundaryEdge.Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {
    
            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;
    
            for (int i = 0; i < length; i++) {
    
                double[] EdgeNormal = LsTrk.GridDat.Edges.NormalsForAffine.ExtractSubArrayShallow(i0 + i, -1).To1DArray();
                if (LsTrk.GridDat.Edges.CellIndices[i0 + i, 1] == jCell) {
                    // Console.WriteLine($"jCell {jCell} OUT on edge {i0 + i} - change direction");
                    EdgeNormal.ScaleV(-1.0);
                }
                // Console.WriteLine($"normal at Edge {i0+i} : ({EdgeNormal[0]}, {EdgeNormal[1]})");
                // Console.WriteLine($"normal at Edge {i0+i} : ({EdgeNormal[0]}, {EdgeNormal[1]}, , {EdgeNormal[2]})");
                    
                // Console.WriteLine($"edge {i0 + i} - number of nodes {qN}");
                int trf = (LsTrk.GridDat.Edges.CellIndices[i0+i, 0] == jCell) ? LsTrk.GridDat.Edges.Edge2CellTrafoIndex[i0 + i, 0] : LsTrk.GridDat.Edges.Edge2CellTrafoIndex[i0 + i, 1];
                NodeSet VolNodes = QR.Nodes.GetVolumeNodeSet(LsTrk.GridDat, trf, false);
                // Console.WriteLine($"volume node: ({VolNodes[0, 0]}, {VolNodes[0, 1]})");  
        
                var LsNormals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(VolNodes, jCell, length);
                // var phiGrad = MultidimensionalArray.Create(length, qN, D);
                // ((LevelSet)LsTrk.LevelSets[0]).EvaluateGradient(jCell, length, VolNodes, phiGrad);
                    
                for (int d = 0; d < D; d++) {
    
                    var fEval = MultidimensionalArray.Create(length, qN);
                    testField[d].Evaluate(jCell, length, VolNodes, fEval);
    
                    for (int qn = 0; qn < qN; qn++) {
                        // Console.WriteLine($"LevelSet normal at Edge {i0+i}: ({LsNormals[i, qn, 0]}, {LsNormals[i, qn, 1]})");
                        // Console.WriteLine($"LevelSet normal at Edge {i0+i}: ({LsNormals[i, qn, 0]}, {LsNormals[i, qn, 1]}, {LsNormals[i, qn, 2]})");
                        // double[] LsNormal = new double[] { phiGrad[i, qn, 0], phiGrad[i, qn, 1] };
                        // LsNormal.Normalize();
                        // Console.WriteLine($"PhiGrad normal at Edge {i0+i}: ({LsNormal[0]}, {LsNormal[1]})");

                        double[] LsTangent = new double[D];
                        for (int d1 = 0; d1 < D; d1++) {
                            for (int d2 = 0; d2 < D; d2++) {
                                double nn = LsNormals[i, qn, d1] * LsNormals[i, qn, d2];
                                if (d1 == d2) {
                                    LsTangent[d1] += (1 - nn) * EdgeNormal[d2];
                                } else {
                                    LsTangent[d1] += -nn * EdgeNormal[d2];
                                }
                            }
                        }         
                        LsTangent = LsTangent.Normalize();

                        // double tangentCheck = GenericBlas.InnerProd(LsTangent, LsNormals.ExtractSubArrayShallow(i, qn, -1).To1DArray());
                        // Console.WriteLine($"LsTangent * LsNormals = {tangentCheck}");                       
    
                        // if (D != 2) 
                        //     throw new ArgumentException();
    
                        // double[] LsTangent = new double[2];
                        
                        // LsTangent[0] = -LsNormals[i, qn, 1];
                        // LsTangent[1] = LsNormals[i, qn, 0];
            
                        // if (GenericBlas.InnerProd(LsTangent, EdgeNormal) < 0.0) {
                        //     // Console.WriteLine($"LsTangent change direction");
                        //     // LsTangent.ScaleV(-1.0);
                        //     Console.WriteLine($"LsTangent NOT pointing outward!");
                        // }
    
                        // Console.WriteLine($"LS tangent: ({LsTangent[0]}, {LsTangent[1]})");  
                        // Console.WriteLine($"LS tangent: ({LsTangent[0]}, {LsTangent[1]} , {LsTangent[2]})");  
                        EvalResult[i, qn, 0] += LsTangent[d] * fEval[i, qn];
                    }
                }
            }
    
        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                stokesAtEdges += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();

    // =========================================================
    double error = stokesAtInterface + stokesAtEdges;

    return (error, stokesAtInterface, stokesAtEdges);
}

In [ ]:
List<double> StokesInCutCellsOverTime = new List<double>();
if (checkRange) {
    foreach(var ts in timesteps) {

        var ts_LsTrk = GetLsTrkForTimestep(ts, mfv);

        int degU = ts.GetField("VelocityX").Basis.Degree;
        int ts_order = GetXNSEquadOrder(degU, mfv);
       
        var testField = GetTestField(ts_LsTrk.GridDat);

        double totalStokesErrorInCC = 0.0;
        foreach (int jC in ts_LsTrk.Regions.GetCutCellMask().ItemEnum) {
            var result = CheckStokesForCell(jC, ts_LsTrk, ts_LsTrk.GetSpeciesId("A"), testField, ts_order);
            totalStokesErrorInCC += result.error.Abs();
        }

        StokesInCutCellsOverTime.Add(totalStokesErrorInCC);
    }
}

In [ ]:
StokesInCutCellsOverTime

index,value
0,0.01608522934501162
1,0.014156438598209224
2,0.01456819903652023


In [ ]:
var LsTrk = GetLsTrkForTimestep(tStep, mfv);

var testField = GetTestField(LsTrk.GridDat);

int degU = tStep.GetField("VelocityX").Basis.Degree;
int order = GetXNSEquadOrder(degU, mfv);
order

19

In [ ]:
double totalStokesError = 0.0;
SinglePhaseField stokesErrorField = new SinglePhaseField(new Basis(LsTrk.GridDat, 0), "stokesError");
foreach (int jC in LsTrk.Regions.GetCutCellMask().ItemEnum) {
    var result = CheckStokesForCell(jC, LsTrk, LsTrk.GetSpeciesId("A"), testField, order);
    Console.WriteLine($"Stokes error in Cell {jC}: {result.error}");
    stokesErrorField.SetMeanValue(jC, result.error);
    totalStokesError += result.error.Abs();
}
totalStokesError

Stokes error in Cell 145: 7.835644276056586E-06
Stokes error in Cell 146: -3.558610225025318E-06
Stokes error in Cell 147: 2.087672108824659E-07
Stokes error in Cell 152: -1.0060667409935134E-06
Stokes error in Cell 153: 2.1780748476619982E-05
Stokes error in Cell 154: 8.25047154027884E-06
Stokes error in Cell 155: 6.079880787128651E-06
Stokes error in Cell 160: -1.124312494307108E-05
Stokes error in Cell 161: 2.8451052145423588E-05
Stokes error in Cell 162: 5.010272421094371E-06
Stokes error in Cell 163: -1.1223421409102869E-05
Stokes error in Cell 168: 0.0028491822284903378
Stokes error in Cell 169: -0.0030249769666881613
Stokes error in Cell 170: -1.5594507597707438E-05
Stokes error in Cell 171: 2.24147028364635E-06
Stokes error in Cell 208: -6.071978085081361E-06
Stokes error in Cell 209: 2.1077283867346717E-05
Stokes error in Cell 210: 2.668429471642652E-06
Stokes error in Cell 211: 7.39205395280571E-07
Stokes error in Cell 216: -3.3887382343018163E-06
Stokes error in Cell 217: 5.

0.01456819903652023

In [ ]:
int jCell = 136;
CheckStokesForCell(jCell, LsTrk, LsTrk.GetSpeciesId("A"), testField, order)

Item1,Item2,Item3
0.039922601027772675,-1.1215178936960104,1.161440494723783


In [ ]:
SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
levelSet.Acc(1.0, phi); 
SinglePhaseField curvatureField = new SinglePhaseField(new Basis(LsTrk.GridDat, phi.Basis.Degree - 2), "Curvature");
levelSet.ComputeTotalCurvature(curvatureField, LsTrk.Regions.GetCutCellMask());
curvatureField.Scale(0.5);  // mean curvature

In [ ]:
DGField[] plotFields = new DGField[] { phi, gaussErrorField, stokesErrorField, curvatureField };
Tecplot("IntegralErrors3D", tStep.PhysicalTime, 3, plotFields)

Writing output file d:\BoSSS-experimental\public\examples\RisingBubble\IntegralErrors3D...
done.


## check quadrature rules

In [ ]:
public static (int VolNodesCount, int EdgeNodesCount) CheckQuadratueRulesInCutCell(int jCell, LevelSetTracker LsTrk, int order) {

    var grdDat = LsTrk.GridDat;
    BitArray cellIntDomBA = new BitArray(grdDat.Cells.NoOfLocalUpdatedCells);
    cellIntDomBA[jCell] = true;
    CellMask cellIntDom = new CellMask(grdDat, cellIntDomBA);

    XQuadSchemeHelper schemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;
    

    // check LevelSetQuad
    var VolQuadScheme = schemeHelper.GetLevelSetquadScheme(0, cellIntDom, order);
    var VQR = VolQuadScheme.Compile(grdDat, order).ElementAt(0).Rule;
    int VolNodesCount = VQR.NoOfNodes;


    // check EdgeQuad
    BitArray edgeIntDomBA = new BitArray(grdDat.iGeomEdges.Count);
    var jCell2Edges = grdDat.Cells.Cells2Edges[jCell];
    foreach (int iE in jCell2Edges) {
        if (iE < 0) 
            edgeIntDomBA[(-iE) - 1] = true;
        else 
            edgeIntDomBA[iE - 1] = true;
    }
    EdgeMask edgeIntDom = new EdgeMask(grdDat, edgeIntDomBA);
    var EdgeQuadScheme = schemeHelper.GetEdgeQuadScheme(LsTrk.GetSpeciesId("A"), IntegrationDomain: edgeIntDom);
    var EQR = EdgeQuadScheme.Compile(grdDat, order).ElementAt(0).Rule;
    int EdgeNodesCount = EQR.NoOfNodes;


    return (VolNodesCount, EdgeNodesCount);
}

In [ ]:
var tStep = sess.Timesteps.TakeLast(2).Last();
tStep

 { Time-step: 1102; Physical time: 2.201999999999979s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, GravityY#A, GravityY#B, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }

In [ ]:
var LsTrk = GetLsTrkForTimestep(tStep, mfv);

var testField = GetTestField(LsTrk.GridDat);

int degU = tStep.GetField("VelocityX").Basis.Degree;
int order = GetXNSEquadOrder(degU, mfv);
order

19

In [ ]:
SinglePhaseField VolQuadNodesField = new SinglePhaseField(new Basis(LsTrk.GridDat, 0), "VolumeQuadNodes");
foreach (int jC in LsTrk.Regions.GetCutCellMask().ItemEnum) {
    var result = CheckQuadratueRulesInCutCell(jC, LsTrk, order);
    Console.WriteLine($"Quadrature nodes in Cell {jC}: volume nodes {result.VolNodesCount}, edge nodes {result.EdgeNodesCount}");
    VolQuadNodesField.SetMeanValue(jC, result.VolNodesCount);
}

Quadrature nodes in Cell 135: volume nodes 10, edge nodes 10
Quadrature nodes in Cell 136: volume nodes 10, edge nodes 10
Quadrature nodes in Cell 137: volume nodes 20, edge nodes 10
Quadrature nodes in Cell 138: volume nodes 10, edge nodes 10
Quadrature nodes in Cell 139: volume nodes 10, edge nodes 10
Quadrature nodes in Cell 175: volume nodes 20, edge nodes 10
Quadrature nodes in Cell 179: volume nodes 10, edge nodes 10
Quadrature nodes in Cell 180: volume nodes 10, edge nodes 10
Quadrature nodes in Cell 215: volume nodes 10, edge nodes 10
Quadrature nodes in Cell 220: volume nodes 20, edge nodes 10
Quadrature nodes in Cell 221: volume nodes 10, edge nodes 10
Quadrature nodes in Cell 255: volume nodes 10, edge nodes 10
Quadrature nodes in Cell 261: volume nodes 10, edge nodes 10
Quadrature nodes in Cell 295: volume nodes 10, edge nodes 10
Quadrature nodes in Cell 301: volume nodes 10, edge nodes 10
Quadrature nodes in Cell 302: volume nodes 20, edge nodes 10
Quadrature nodes in Cell

In [ ]:
DGField[] plotFields = new DGField[] { tStep.GetField("Phi"), VolQuadNodesField };
Tecplot("QuadNodes", tStep.PhysicalTime, 3, plotFields)

Writing output file d:\BoSSS-experimental\public\examples\RisingBubble\QuadNodes...
done.


## Evaluate Interface Continuity

In [ ]:
int order = phiCP.Basis.Degree;
XQuadSchemeHelper SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;
EdgeQuadratureScheme SurfaceElement_Edge = SchemeHelper.Get_SurfaceElement_EdgeQuadScheme(LsTrk.GetSpeciesId("A"), 0);

In [ ]:
//SurfaceElement_Edge.Domain.ItemEnum.Count()

In [ ]:
EdgeMask CutCellBoundaryEdgeMask = LsTrk.Regions.GetCutCellMask().AllEdges().Except(LsTrk.Regions.GetCutCellMask().GetAllInnerEdgesMask());
//CutCellBoundaryEdgeMask.ItemEnum.Count()

In [ ]:
var factory = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadFactoryHelper.GetSurfaceElement_BoundaryRuleFactory(0, LsTrk.GridDat.Grid.RefElements[0]);
EdgeQuadratureScheme SurfaceElement_BoundaryEdge = new EdgeQuadratureScheme(factory, CutCellBoundaryEdgeMask);

In [ ]:
double result = 0;
int D = LsTrk.GridDat.SpatialDimension;
EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    SurfaceElement_BoundaryEdge.Compile(LsTrk.GridDat, order),
    delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

        // inner quadrature node
        NodeSet Enode_l = QR.Nodes;
        int trf = LsTrk.GridDat.Edges.Edge2CellTrafoIndex[i0, 0];
        NodeSet Vnode_l = Enode_l.GetVolumeNodeSet(LsTrk.GridDat, trf, false);
        NodeSet Vnode_g = Vnode_l.CloneAs();
        int cell = LsTrk.GridDat.Edges.CellIndices[i0, 0];
        LsTrk.GridDat.TransformLocal2Global(Vnode_l, Vnode_g, cell);
        if (D == 2)
            Console.WriteLine("inner quadrature node: ({0},{1})", Vnode_g[0, 0], Vnode_g[0, 1]);
        else 
            Console.WriteLine("inner quadrature node: ({0},{1},{2})", Vnode_g[0, 0], Vnode_g[0, 1], Vnode_g[0, 2]);

        // for(int i = 0; i < length; i++) {
        //     EvalResult[i, 0, 0] = 1;    
        // }
        EvalResult.SetAll(1.0);

    },
    delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
        for(int i = 0; i < length; i++) {
            result += ResultsOfIntegration[i, 0];
        }
    }
).Execute();
result

0

In [ ]:
CellQuadratureScheme ContactLineVolumeScheme = SchemeHelper.GetContactLineQuadScheme(LsTrk.GetSpeciesId("A"), 0);

In [ ]:
ContactLineVolumeScheme.Domain.ItemEnum.Count()

0